# Check an Analyst report

Copy this file into `private/reviews/` and rename the copy to `<packet_id>_review.ipynb`
(for example, `EP_0003_review.ipynb`). Work in that copy; keep this template blank.

1. Set `attempt_id` below to the folder name printed by generation.
2. Run the cells and read each claim beside its evidence.
3. Fill in **Notes / feedback** and make any fixes in the edit cell.
4. When finished, set `ready = True`, run all cells again, and save the notebook.

The last cell saves the final report to public. The original candidate stays unchanged.
There are no model calls here. Leave `ready = False` while checking.


In [14]:
import copy
import json
import sys
from decimal import Decimal
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/benchmark/private/packets").is_dir())
sys.path.insert(0, str(repo))
from src.benchmark.prepare import read, save, mask, assert_masked, check_report
from src.benchmark.schemas import Packet, Report

folder = repo / "data/benchmark"
attempt_id = "60253b6e5a954d518737d3e97439392c"  # change this in your copy
attempt_folder = folder / "private/attempts" / attempt_id
attempt = read(attempt_folder / "attempt.json")
sent_packet = json.loads(attempt["request"]["messages"][1]["content"])["evidence_packet"]
original_report = read(attempt_folder / "candidate.json")
aliases = read(folder / "private/aliases.json")
packet = mask(sent_packet, aliases)
report = copy.deepcopy(original_report)
print("Packet:", packet["packet_id"], "| Model:", attempt["request"]["model"])
print("Packet masking changed:", packet != sent_packet)

Packet: EP_0014 | Model: google/gemini-2.5-flash
Packet masking changed: False


In [15]:
print("Recommendation:", report["recommendation"])
evidence = {e["evidence_id"]: e["text"] for e in packet["evidence"]}
for claim in report["claims"]:
    print("\nCLAIM", claim["claim_id"], claim["text"])
    for evidence_id in claim["evidence_ids"]:
        print("EVIDENCE", evidence_id, evidence.get(evidence_id, "MISSING"))
print("\nRationale:", report["rationale"])

Recommendation: SELL

CLAIM C1 Consolidated Sales and Revenues increased by 4.29% from $64,809 million in 2024 to $67,589 million in 2025.
EVIDENCE EP_0014_E01 Sales and Revenues by Segment
(Millions of dollars) 2024 SalesVolume PriceRealization Currency Inter-Segment/Other 2025 $Change %Change
SEGMENT_33 $ 25,455 $ 568 $ (1,136) $ 24 $ 149 $ 25,060 $ (395) (2%)
SEGMENT_34 12,471 403 (272) (46) (82) 12,474 3 —%
SEGMENT_36 28,854 2,401 592 62 292 32,201 3,347 12%
All Other Segment 344 11 — (1) (27) 327 (17) (5%)
Corporate Items and Eliminations (5,761) 6 (1) 6 (332) (6,082) (321)  
SEGMENT_35 61,363 3,389 (817) 45 — 63,980 2,617 4%
SEGMENT_37 4,053 — — — 167 4,220 167 4%
Corporate Items and Eliminations (607) — — — (4) (611) (4)  
SEGMENT_37 3,446 — — — 163 3,609 163 5%
Consolidated Sales and Revenues $ 64,809 $ 3,389 $ (817) $ 45 $ 163 $ 67,589 $ 2,780 4%

CLAIM C2 Despite the increase in revenue, Consolidated Operating Profit decreased by 14.70% from $13,072 million in 2024 to $11,151

## Numbers

Compare the claimed amount or percentage with the calculation. Also read the prose above:
these checks only cover the structured assertion, not every number in a sentence.

In [16]:
facts_by_id = {f["fact_id"]: f for f in packet["facts"]}
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        print(claim["claim_id"], "Check against the excerpt above.")
        continue
    ids = [spec.get("fact_id")] if spec["kind"] == "numeric" else [spec.get("from_fact_id"), spec.get("to_fact_id")]
    periods = [spec["period"]] if spec["kind"] == "numeric" else [spec["from_period"], spec["to_period"]]
    if all(ids):
        selected = [facts_by_id.get(fact_id) for fact_id in ids]
        print(claim["claim_id"], "Referenced facts:")
        pprint(selected)
        if any(f is None for f in selected):
            print("Unknown fact ID; check manually.")
            continue
    else:
        print(claim["claim_id"], "Older candidate: references are added in the edit cell below.")
        matches = [[f for f in packet["facts"] if f["metric"] == spec["metric"]
                    and f["evidence_id"] in claim["evidence_ids"] and f["period"] == period
                    and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])] for period in periods]
        if any(len(found) != 1 for found in matches):
            print("No exact fact match; check manually.")
            continue
        selected = [found[0] for found in matches]
        pprint(selected)
    values = [Decimal(f["value"]) for f in selected]
    if spec["kind"] == "percentage_change" and values[0] == 0:
        print(claim["claim_id"], "Cannot calculate percentage change from zero.")
        continue
    expected = values[0] if spec["kind"] == "numeric" else (values[1] - values[0]) / values[0] * 100
    print(claim["claim_id"], "claimed:", spec["value"], "calculated:", expected,
          "rounded:", expected.quantize(Decimal("0.01")))
pprint(check_report(Packet.model_validate(packet), Report.model_validate(report)))


C1 Referenced facts:
[{'evidence_id': 'EP_0014_E01',
  'fact_id': 'EP_0014_F02',
  'metric': 'revenue',
  'period': {'end': '2024-12-31', 'label': 'FY2024', 'start': '2024-01-01'},
  'unit': 'USD_million',
  'value': '64809'},
 {'evidence_id': 'EP_0014_E01',
  'fact_id': 'EP_0014_F01',
  'metric': 'revenue',
  'period': {'end': '2025-12-31', 'label': 'FY2025', 'start': '2025-01-01'},
  'unit': 'USD_million',
  'value': '67589'}]
C1 claimed: 4.29 calculated: 4.289527689055532410621981515 rounded: 4.29
C2 Referenced facts:
[{'evidence_id': 'EP_0014_E02',
  'fact_id': 'EP_0014_F04',
  'metric': 'operating_profit',
  'period': {'end': '2024-12-31', 'label': 'FY2024', 'start': '2024-01-01'},
  'unit': 'USD_million',
  'value': '13072'},
 {'evidence_id': 'EP_0014_E02',
  'fact_id': 'EP_0014_F03',
  'metric': 'operating_profit',
  'period': {'end': '2025-12-31', 'label': 'FY2025', 'start': '2025-01-01'},
  'unit': 'USD_million',
  'value': '11151'}]
C2 claimed: -14.70 calculated: -14.69553243

## Notes / feedback

Replace the blanks with short notes. Use the evidence-checking notebook to check the source PDF too.

- Source excerpts and masking: ...
- Claims, numbers, years, units and citations: ...
- Reasoning and risks / counter-evidence: ...
- Problems found and what I changed (or no changes needed): C5 percentage is slightly wrong, shd be  −24.17%, not −24.18%.
- Decision: checked yet.


In [17]:
# Make any corrections here. Skip this if no changes are needed.
# Example: report["claims"][0]["text"] = "Corrected claim..."
# If you correct a number, update its check_spec too, if present.
# Keep the edits in this cell so running the notebook again applies them.

report["claims"][4]["text"] = report["claims"][4]["text"].replace(
    "24.18%", "24.17%"
)
report["claims"][4]["check_spec"]["value"] = "-24.17"
# Add explicit source-fact references to the reviewed copy of older candidates.
# This does not change the saved model response, numbers, or wording.
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        continue
    fields = [("fact_id", "period")] if spec["kind"] == "numeric" else [("from_fact_id", "from_period"), ("to_fact_id", "to_period")]
    for id_field, period_field in fields:
        if spec.get(id_field):
            continue
        matches = [f for f in packet["facts"] if f["metric"] == spec["metric"]
                   and f["period"] == spec[period_field] and f["evidence_id"] in claim["evidence_ids"]
                   and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])]
        assert len(matches) == 1, "Check the fact mapping manually: " + claim["claim_id"]
        spec[id_field] = matches[0]["fact_id"]

pprint(report, sort_dicts=False)


{'schema_version': '0.1',
 'report_id': 'R_60253b6e5a954d518737d3e97439392c',
 'recommendation': 'SELL',
 'claims': [{'claim_id': 'C1',
             'text': 'Consolidated Sales and Revenues increased by 4.29% from '
                     '$64,809 million in 2024 to $67,589 million in 2025.',
             'evidence_ids': ['EP_0014_E01'],
             'check_spec': {'kind': 'percentage_change',
                            'from_fact_id': 'EP_0014_F02',
                            'to_fact_id': 'EP_0014_F01',
                            'metric': 'revenue',
                            'from_period': {'label': 'FY2024',
                                            'start': '2024-01-01',
                                            'end': '2024-12-31'},
                            'to_period': {'label': 'FY2025',
                                          'start': '2025-01-01',
                                          'end': '2025-12-31'},
                            'value': '4.29',
         

## Save the final copy

Set `ready = True` only when the checks and edits are finished.
This checks the edited report and saves it with the masked packet to `public/`.
It replaces existing files with the same IDs, so use this for the working dataset.
Keep released dataset versions unchanged.

Save this notebook with its outputs. These notes are also the feedback;
there is no separate review JSON or feedback file to fill in.


The report is saved as `public/reports/<packet_id>/clean.json`.
For older candidates, the edit cell adds fact IDs only when metric, period, unit and evidence match uniquely. Check the printed references before saving.

In [18]:
ready = True
if ready:
    final_report = Report.model_validate(report)
    final_packet = Packet.model_validate(packet)
    checks = check_report(final_packet, final_report, require_fact_ids=True)
    assert not checks["errors"], checks["errors"]
    assert_masked(report, aliases)
    assert_masked(packet, aliases)
    save(folder / "public/evidence_packets" / (packet["packet_id"] + ".json"), final_packet.model_dump(mode="json"))
    output = folder / "public/reports" / packet["packet_id"] / "clean.json"
    save(output, final_report.model_dump(mode="json"))
    print("Saved:", output)
else:
    print("Not saved yet.")

Saved: /Users/claudia/Desktop/dsa4213/dsa4213 proj/DSA4213_Group11/data/benchmark/public/reports/EP_0014/clean.json
